<a href="https://colab.research.google.com/github/bored-shinigami2805/CVPR-SLM-fine-tuning/blob/main/chatLoop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" bitsandbytes gradio
import torch; print("cuda available:", torch.cuda.is_available())

In [ ]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_ID     = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "/content/qwen2p5-3b-cv-lora"   # or None for the base model
USE_4BIT    = True

SYS = ("You are a computer-vision research assistant. You have studied a specific set of "
       "computer-vision papers in depth. Answer accurately and concisely, grounded only in "
       "what you know. If a question is outside computer vision, say it is outside your domain. "
       "If it is about computer vision but not something you have studied, say you don't have "
       "that information rather than guessing.")

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR if ADAPTER_DIR and os.path.isdir(ADAPTER_DIR) else BASE_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

quant = None
if USE_4BIT:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, quantization_config=quant, torch_dtype=torch.bfloat16, device_map="auto")
if ADAPTER_DIR and os.path.isdir(ADAPTER_DIR):
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    print("loaded LoRA adapter from", ADAPTER_DIR)
else:
    print("running BASE model (no adapter)")
model.eval(); model.config.use_cache = True

In [ ]:
import threading
import gradio as gr
from transformers import TextIteratorStreamer

def respond(message, history):
    # Build the full conversation, robust to either gradio history format.
    msgs = [{"role": "system", "content": SYS}]
    for h in (history or []):
        if isinstance(h, dict) and h.get("content"):
            msgs.append({"role": h.get("role", "user"), "content": h["content"]})
        elif isinstance(h, (list, tuple)) and len(h) == 2:
            u, a = h
            if u: msgs.append({"role": "user", "content": u})
            if a: msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": message})

    enc = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kw = dict(**enc, max_new_tokens=512, do_sample=False, streamer=streamer,
              pad_token_id=tok.pad_token_id, repetition_penalty=1.05)

    err = {}
    def _run():
        try:
            with torch.no_grad():
                model.generate(**kw)
        except Exception as e:          # never let the streamer hang the UI
            err["e"] = e
            streamer.end()
    threading.Thread(target=_run, daemon=True).start()

    partial = ""
    for piece in streamer:
        partial += piece
        yield partial
    if err:
        yield partial + f"\n\n[generation error: {err['e']}]"

demo = gr.ChatInterface(
    fn=respond,
    title="CV research assistant (Qwen2.5-3B + LoRA)",
    description="Ask about the studied computer-vision papers.")
demo.launch(share=True, debug=False)